# EMS741 — Few-Shot Medical Image Segmentation
## U-Net baseline + Reptile meta-learning

**Setup:** Change runtime to GPU, then click *Run all*.

### Dataset structure (after extraction)
```
./train/  task_2/  images/ *.png   masks/ *.png
          task_3/  ...
          task_5/  ...
          task_7/  ...
./val/    task_4/  ...
          task_6/  ...
./test/   task_1/  ...
          task_8/  ...
```

## 1. Download & extract dataset

In [ ]:
# Download the dataset from Zenodo (provided by module)
!wget -O data.zip https://zenodo.org/records/18745413/files/ems741_cw_data.zip?download=1

import zipfile, os

path_to_zip = r'data.zip'
path_to_extract_to = r'./'

with zipfile.ZipFile(path_to_zip, 'r') as zip_ref:
    zip_ref.extractall(path_to_extract_to)

# Verify extraction
print('train:', os.listdir('./train'))
print('val:  ', os.listdir('./val'))
print('test: ', os.listdir('./test'))

# Set DATA_ROOT to current directory (data extracts as ./train, ./val, ./test)
DATA_ROOT = '.'
print('\nDATA_ROOT set to:', os.path.abspath(DATA_ROOT))

## 2. Install / import dependencies

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

import random, copy, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)
torch.manual_seed(42)
np.random.seed(42)

## 3. Dataset inspection & mask verification

> **Why masks look black:** The mask PNGs store labels as pixel value `0` (background)
> and `1` (foreground). PIL reads them as-is; `imshow` maps 1→ nearly black.
> The fix is simply `mask * 255` for display, and keeping raw 0/1 for training.

In [ ]:
def discover_tasks(split_dir):
    """Return dict {task_name: {'images': [...], 'masks': [...]}} sorted by name."""
    split_dir = Path(split_dir)
    tasks = {}
    for task_dir in sorted(split_dir.iterdir()):
        if not task_dir.is_dir():
            continue
        imgs  = sorted((task_dir / 'images').glob('*.png'))
        masks = sorted((task_dir / 'masks').glob('*.png'))
        assert len(imgs) == len(masks), f'{task_dir}: img/mask count mismatch'
        tasks[task_dir.name] = {'images': imgs, 'masks': masks}
    return tasks


def load_sample(img_path, mask_path, img_size=256):
    """Load one image-mask pair.

    Returns
    -------
    image : np.ndarray  float32 in [0, 1], shape (H, W)
    mask  : np.ndarray  uint8   in {0, 1}, shape (H, W)
    """
    img  = Image.open(img_path).convert('L').resize((img_size, img_size), Image.BILINEAR)
    mask = Image.open(mask_path).convert('L').resize((img_size, img_size), Image.NEAREST)

    img_arr  = np.array(img,  dtype=np.float32) / 255.0

    mask_arr = np.array(mask, dtype=np.uint8)
    # ── KEY FIX: normalise mask to binary 0/1 regardless of storage convention ──
    # Masks stored as {0,1}: unique values are {0,1} → no change needed
    # Masks stored as {0,255}: threshold to get {0,1}
    if mask_arr.max() > 1:
        mask_arr = (mask_arr > 127).astype(np.uint8)
    # If mask is all zeros, still valid — structure absent from this slice

    return img_arr, mask_arr


# ── Discover tasks ──
train_tasks = discover_tasks(Path(DATA_ROOT) / 'train')  # ./train/task_2, task_3, ...
val_tasks   = discover_tasks(Path(DATA_ROOT) / 'val')    # ./val/task_4, task_6
test_tasks  = discover_tasks(Path(DATA_ROOT) / 'test')   # ./test/task_1, task_8

print('Train tasks:', list(train_tasks.keys()))
print('Val tasks  :', list(val_tasks.keys()))
print('Test tasks :', list(test_tasks.keys()))

# ── Sanity-check: print mask statistics for first task ──
first_task = next(iter(train_tasks.values()))
print('\n── Mask pixel-value audit (first 5 samples of first train task) ──')
for i in range(min(5, len(first_task['masks']))):
    arr = np.array(Image.open(first_task['masks'][i]).convert('L'))
    print(f'  sample {i}: unique values={np.unique(arr)}, '
          f'min={arr.min()}, max={arr.max()}, '
          f'nonzero pixels={np.count_nonzero(arr)}')

In [ ]:
# ── Visualise 3 image-mask pairs from the first training task ──
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
fig.suptitle('Image (top) and binary mask (bottom) — verify masks are non-empty', fontsize=13)

for col in range(3):
    idx = col * max(1, len(first_task['images']) // 3)
    img, mask = load_sample(first_task['images'][idx], first_task['masks'][idx])

    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f'Image slice {idx}')
    axes[0, col].axis('off')

    # mask * 255 so white = foreground
    axes[1, col].imshow(mask * 255, cmap='gray', vmin=0, vmax=255)
    pct = mask.mean() * 100
    axes[1, col].set_title(f'Mask — {pct:.1f}% foreground')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

print('\nIf masks above are non-empty → dataset is correct, proceed.')
print('If all masks are black    → check DATA_ROOT path or re-download dataset.')

## 4. Dataset classes

In [ ]:
IMG_SIZE = 256


class SegDataset(Dataset):
    """Simple single-task segmentation dataset."""

    def __init__(self, task_dict, augment=False):
        self.images  = task_dict['images']
        self.masks   = task_dict['masks']
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img, mask = load_sample(self.images[idx], self.masks[idx], IMG_SIZE)

        # Convert to tensors  (C, H, W)
        img_t  = torch.from_numpy(img).unsqueeze(0)           # (1, H, W)
        mask_t = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0)  # (1, H, W)

        if self.augment:
            if random.random() > 0.5:
                img_t  = TF.hflip(img_t)
                mask_t = TF.hflip(mask_t)
            if random.random() > 0.5:
                img_t  = TF.vflip(img_t)
                mask_t = TF.vflip(mask_t)
            angle = random.uniform(-15, 15)
            img_t  = TF.rotate(img_t,  angle)
            mask_t = TF.rotate(mask_t, angle)

        return img_t, mask_t


class FewShotEpisode(Dataset):
    """Returns exactly `n_shot` support samples and all remaining as query."""

    def __init__(self, task_dict, n_shot, seed=42):
        rng = random.Random(seed)
        all_idx = list(range(len(task_dict['images'])))
        support_idx = rng.sample(all_idx, n_shot)
        query_idx   = [i for i in all_idx if i not in support_idx]

        self.support = {
            'images': [task_dict['images'][i] for i in support_idx],
            'masks' : [task_dict['masks'][i]  for i in support_idx],
        }
        self.query = {
            'images': [task_dict['images'][i] for i in query_idx],
            'masks' : [task_dict['masks'][i]  for i in query_idx],
        }

    def support_loader(self, batch_size=None):
        ds = SegDataset(self.support, augment=False)
        bs = batch_size or len(ds)
        return DataLoader(ds, batch_size=bs, shuffle=True)

    def query_loader(self, batch_size=4):
        ds = SegDataset(self.query, augment=False)
        return DataLoader(ds, batch_size=batch_size, shuffle=False)


print('Dataset classes defined.')

## 5. Model — lightweight U-Net

In [ ]:
class DoubleConv(nn.Module):
    """Two 3×3 Conv-BN-ReLU blocks."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    """Compact U-Net: 1-channel grayscale in, 1-channel sigmoid mask out.

    Channel widths [32, 64, 128, 256] give ~7M parameters — manageable on Colab.
    """

    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2)

        in_ch = 1
        for ch in channels:
            self.downs.append(DoubleConv(in_ch, ch))
            in_ch = ch

        self.bottleneck = DoubleConv(channels[-1], channels[-1] * 2)

        rev = list(reversed(channels))
        for ch in rev:
            self.ups.append(nn.ConvTranspose2d(ch * 2, ch, 2, stride=2))
            self.ups.append(DoubleConv(ch * 2, ch))

        self.out_conv = nn.Conv2d(channels[0], 1, 1)

    def forward(self, x):
        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skip_connections[i // 2]
            if x.shape != skip.shape:  # handle rounding
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i + 1](x)

        return torch.sigmoid(self.out_conv(x))


# Quick parameter count
model = UNet().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'U-Net parameters: {n_params:,}')

# Smoke-test forward pass
with torch.no_grad():
    dummy = torch.randn(2, 1, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out   = model(dummy)
    print(f'Input shape: {dummy.shape}  →  Output shape: {out.shape}')

## 6. Loss & metric

In [ ]:
def dice_loss(pred, target, smooth=1.0):
    """Soft Dice loss. pred and target are both in [0, 1], shape (B, 1, H, W)."""
    pred   = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1.0 - (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)


def bce_dice_loss(pred, target, bce_weight=0.5):
    """Combined BCE + Dice loss (common in medical segmentation)."""
    bce  = F.binary_cross_entropy(pred, target)
    dice = dice_loss(pred, target)
    return bce_weight * bce + (1 - bce_weight) * dice


@torch.no_grad()
def dice_score(pred, target, threshold=0.5, smooth=1.0):
    """Hard Dice score for evaluation. Returns float."""
    pred_bin = (pred > threshold).float()
    intersection = (pred_bin * target).sum()
    return ((2.0 * intersection + smooth) / (pred_bin.sum() + target.sum() + smooth)).item()


print('Loss and metric functions defined.')

## 7. Baseline — standard supervised fine-tuning

The baseline trains a U-Net from scratch on only `N_SHOT` labelled examples from
the test task. This represents the naive approach with limited data.

In [ ]:
def train_baseline(task_dict, n_shot, n_epochs=50, lr=1e-3, seed=42):
    """Train a fresh U-Net on n_shot examples from task_dict.

    Returns
    -------
    model        : trained UNet
    history      : dict with 'train_loss' and 'val_dice' lists
    episode      : FewShotEpisode (for query evaluation)
    """
    episode = FewShotEpisode(task_dict, n_shot, seed=seed)
    support_loader = episode.support_loader(batch_size=min(n_shot, 4))
    query_loader   = episode.query_loader(batch_size=4)

    model     = UNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)

    history = {'train_loss': [], 'val_dice': []}

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        for imgs, masks in support_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            preds = model(imgs)
            loss  = bce_dice_loss(preds, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        # Evaluate on query set
        model.eval()
        dice_vals = []
        for imgs, masks in query_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            dice_vals.append(dice_score(preds, masks))
        avg_dice = float(np.mean(dice_vals))

        history['train_loss'].append(epoch_loss / len(support_loader))
        history['val_dice'].append(avg_dice)

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{n_epochs}  '
                  f'loss={history["train_loss"][-1]:.4f}  '
                  f'Dice={avg_dice:.4f}')

    return model, history, episode


print('Baseline training function defined.')

## 8. Few-shot method — Reptile meta-learning

**Algorithm (Nichol et al., 2018):**

```
Initialise θ
for outer iteration 1..T:
    Sample task τ from training tasks
    Compute φ = SGD_k(θ, D_τ)   ← k inner steps on task
    Update θ ← θ + ε (φ − θ)   ← Reptile step
```

The meta-learned initialisation θ* is then fine-tuned on `n_shot` examples of a
new task, requiring far fewer gradient steps than random initialisation.

In [ ]:
def reptile_meta_train(
    train_tasks,
    val_tasks,
    n_outer        = 2000,
    k_inner        = 5,
    inner_lr       = 1e-3,
    meta_lr        = 0.1,
    batch_size     = 4,
    val_every      = 200,
    n_shot_val     = 5,
):
    """Reptile meta-training loop.

    Parameters
    ----------
    train_tasks : dict   {task_name: {'images': [...], 'masks': [...]}}
    val_tasks   : dict   same format, used for monitoring
    n_outer     : int    number of outer (meta) iterations
    k_inner     : int    inner gradient steps per task sample
    inner_lr    : float  learning rate for inner loop SGD
    meta_lr     : float  Reptile step size ε
    batch_size  : int    batch size for inner updates
    val_every   : int    evaluate on val tasks every N outer steps
    n_shot_val  : int    shots used for val-time fine-tuning

    Returns
    -------
    meta_model : UNet with meta-learned weights
    history    : dict with 'outer_step', 'val_dice'
    """
    meta_model = UNet().to(DEVICE)
    task_names = list(train_tasks.keys())
    history    = {'outer_step': [], 'val_dice': []}

    print(f'Reptile meta-training: {n_outer} outer steps, '
          f'{k_inner} inner steps, meta_lr={meta_lr}')

    for outer_step in range(1, n_outer + 1):
        # ── Sample a random training task ──
        task_name = random.choice(task_names)
        task_dict = train_tasks[task_name]
        dataset   = SegDataset(task_dict, augment=True)

        # ── Clone meta parameters for inner loop ──
        fast_model = copy.deepcopy(meta_model)
        inner_opt  = torch.optim.SGD(fast_model.parameters(), lr=inner_lr)

        # ── Inner loop: k gradient steps on randomly sampled batch ──
        fast_model.train()
        for _ in range(k_inner):
            indices = random.sample(range(len(dataset)),
                                    min(batch_size, len(dataset)))
            batch   = [dataset[i] for i in indices]
            imgs    = torch.stack([b[0] for b in batch]).to(DEVICE)
            masks   = torch.stack([b[1] for b in batch]).to(DEVICE)
            inner_opt.zero_grad()
            preds = fast_model(imgs)
            loss  = bce_dice_loss(preds, masks)
            loss.backward()
            inner_opt.step()

        # ── Reptile update: move meta-params toward fast-params ──
        with torch.no_grad():
            for meta_p, fast_p in zip(meta_model.parameters(),
                                      fast_model.parameters()):
                meta_p.data += meta_lr * (fast_p.data - meta_p.data)

        # ── Periodic validation ──
        if outer_step % val_every == 0:
            val_dice = evaluate_few_shot(
                meta_model, val_tasks, n_shot=n_shot_val,
                adapt_steps=10, adapt_lr=1e-3
            )
            history['outer_step'].append(outer_step)
            history['val_dice'].append(val_dice)
            print(f'  Step {outer_step:4d}/{n_outer}  '
                  f'val Dice (avg over val tasks) = {val_dice:.4f}')

    return meta_model, history


def adapt_and_evaluate(meta_model, task_dict, n_shot,
                       adapt_steps=20, adapt_lr=1e-3, seed=42):
    """Fine-tune meta_model on n_shot examples; evaluate on remaining query set.

    Returns
    -------
    float : mean Dice on query set
    adapted_model : fine-tuned copy of meta_model
    episode : FewShotEpisode for later visualisation
    """
    episode = FewShotEpisode(task_dict, n_shot, seed=seed)
    adapted = copy.deepcopy(meta_model)
    opt     = torch.optim.Adam(adapted.parameters(), lr=adapt_lr)

    support_loader = episode.support_loader(batch_size=min(n_shot, 4))

    adapted.train()
    for _ in range(adapt_steps):
        for imgs, masks in support_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            loss = bce_dice_loss(adapted(imgs), masks)
            loss.backward()
            opt.step()

    adapted.eval()
    dice_vals = []
    for imgs, masks in episode.query_loader(batch_size=4):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        dice_vals.append(dice_score(adapted(imgs), masks))

    return float(np.mean(dice_vals)), adapted, episode


def evaluate_few_shot(meta_model, tasks, n_shot, adapt_steps=10, adapt_lr=1e-3):
    """Average Dice over all tasks in `tasks` under n_shot adaptation."""
    scores = []
    for task_dict in tasks.values():
        d, _, _ = adapt_and_evaluate(meta_model, task_dict,
                                      n_shot, adapt_steps, adapt_lr)
        scores.append(d)
    return float(np.mean(scores))


print('Reptile functions defined.')

## 9. Run meta-training

In [ ]:
# ── Hyper-parameters ──────────────────────────────────────────────────────────
N_OUTER   = 2000   # outer Reptile iterations (increase to 5000 for better results)
K_INNER   = 5      # inner gradient steps
META_LR   = 0.1    # Reptile step size
INNER_LR  = 1e-3   # inner-loop learning rate
# ─────────────────────────────────────────────────────────────────────────────

meta_model, meta_history = reptile_meta_train(
    train_tasks,
    val_tasks,
    n_outer   = N_OUTER,
    k_inner   = K_INNER,
    inner_lr  = INNER_LR,
    meta_lr   = META_LR,
    val_every = 200,
    n_shot_val= 5,
)

# Save meta-model checkpoint
torch.save(meta_model.state_dict(), 'meta_model.pth')
print('Meta-model saved to meta_model.pth')

In [ ]:
# ── Plot meta-training progress ──
if meta_history['outer_step']:
    plt.figure(figsize=(7, 4))
    plt.plot(meta_history['outer_step'], meta_history['val_dice'],
             marker='o', linewidth=2, color='steelblue')
    plt.xlabel('Outer step')
    plt.ylabel('Val Dice (5-shot)')
    plt.title('Reptile meta-training — validation Dice')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('meta_training_curve.png', dpi=150)
    plt.show()

## 10. Experiment — compare Reptile vs baseline across N-shot settings

**Research question:** How does segmentation performance change as the number of
available annotated examples increases? (1-shot, 3-shot, 5-shot)

In [ ]:
N_SHOTS        = [1, 3, 5]     # few-shot settings to compare
N_REPEATS      = 3             # different random seeds for stability estimate
ADAPT_STEPS    = 20            # fine-tuning steps at test time
BASELINE_EPOCHS= 100           # epochs to train baseline from scratch

results = {}  # {n_shot: {'reptile': [...], 'baseline': [...]}}

for n_shot in N_SHOTS:
    reptile_scores  = []
    baseline_scores = []

    for seed in range(N_REPEATS):
        print(f'\n── {n_shot}-shot | seed {seed} ──')

        # Evaluate across all test tasks
        r_task_scores = []
        b_task_scores = []

        for task_name, task_dict in test_tasks.items():
            # Reptile: adapt meta-model
            r_dice, _, _ = adapt_and_evaluate(
                meta_model, task_dict, n_shot,
                adapt_steps=ADAPT_STEPS, adapt_lr=1e-3, seed=seed
            )
            r_task_scores.append(r_dice)

            # Baseline: train from scratch
            _, b_history, b_ep = train_baseline(
                task_dict, n_shot,
                n_epochs=BASELINE_EPOCHS, lr=1e-3, seed=seed
            )
            b_dice = b_history['val_dice'][-1]
            b_task_scores.append(b_dice)

            print(f'  Task {task_name}: Reptile={r_dice:.4f}  Baseline={b_dice:.4f}')

        reptile_scores.append(float(np.mean(r_task_scores)))
        baseline_scores.append(float(np.mean(b_task_scores)))

    results[n_shot] = {
        'reptile' : reptile_scores,
        'baseline': baseline_scores,
    }

print('\n== Summary ==')
print(f'{"N-shot":>8}  {"Reptile (mean±std)":>22}  {"Baseline (mean±std)":>22}')
for n_shot, v in results.items():
    r = np.array(v['reptile'])
    b = np.array(v['baseline'])
    print(f'{n_shot:>8}  {r.mean():.4f} ± {r.std():.4f}           '
          f'{b.mean():.4f} ± {b.std():.4f}')

## 11. Plot results

In [ ]:
shots   = list(results.keys())
r_means = [np.mean(results[s]['reptile'])  for s in shots]
r_stds  = [np.std(results[s]['reptile'])   for s in shots]
b_means = [np.mean(results[s]['baseline']) for s in shots]
b_stds  = [np.std(results[s]['baseline'])  for s in shots]

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(shots, r_means, yerr=r_stds,
            marker='o', linewidth=2, capsize=5,
            label='Reptile (meta)', color='steelblue')
ax.errorbar(shots, b_means, yerr=b_stds,
            marker='s', linewidth=2, capsize=5, linestyle='--',
            label='Baseline (scratch)', color='coral')
ax.set_xlabel('Number of shots (support examples)', fontsize=12)
ax.set_ylabel('Dice score', fontsize=12)
ax.set_title('Few-shot segmentation: Reptile vs baseline', fontsize=13)
ax.set_xticks(shots)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('results_few_shot.png', dpi=150)
plt.show()

## 12. Qualitative results

In [ ]:
def qualitative_panel(meta_model, task_dict, task_name, n_shot=5,
                      adapt_steps=20, n_cols=4, seed=42):
    """Show image | ground truth | Reptile prediction | Baseline prediction."""
    dice_r, adapted, episode = adapt_and_evaluate(
        meta_model, task_dict, n_shot, adapt_steps=adapt_steps, seed=seed
    )

    # Baseline adapted model
    _, b_hist, _ = train_baseline(task_dict, n_shot, n_epochs=100, seed=seed)

    query_loader = episode.query_loader(batch_size=n_cols)
    imgs_b, masks_b = next(iter(query_loader))
    imgs_b, masks_b = imgs_b.to(DEVICE), masks_b.to(DEVICE)

    adapted.eval()
    with torch.no_grad():
        preds_r = adapted(imgs_b).cpu().numpy()

    imgs_np  = imgs_b.cpu().numpy()
    masks_np = masks_b.cpu().numpy()
    n        = min(n_cols, imgs_np.shape[0])

    fig, axes = plt.subplots(3, n, figsize=(3 * n, 9))
    fig.suptitle(f'Task: {task_name}  |  {n_shot}-shot  |  '
                 f'Reptile Dice = {dice_r:.3f}', fontsize=13)

    row_labels = ['MR image', 'Ground truth', 'Reptile prediction']
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=11)

    for col in range(n):
        axes[0, col].imshow(imgs_np[col, 0], cmap='gray')
        axes[0, col].axis('off')

        axes[1, col].imshow(masks_np[col, 0] * 255, cmap='gray', vmin=0, vmax=255)
        axes[1, col].axis('off')

        axes[2, col].imshow(preds_r[col, 0], cmap='hot', vmin=0, vmax=1)
        axes[2, col].axis('off')

    plt.tight_layout()
    plt.savefig(f'qualitative_{task_name}.png', dpi=150)
    plt.show()


# ── Generate qualitative panels for all test tasks ──
for task_name, task_dict in test_tasks.items():
    qualitative_panel(meta_model, task_dict, task_name, n_shot=5)

## 13. (Optional) Stability analysis

Repeat 5-shot evaluation with different random seeds to measure result variability.

In [ ]:
STABILITY_SEEDS = list(range(10))
N_SHOT_STABILITY = 5

stability_scores = {task: [] for task in test_tasks}

for seed in STABILITY_SEEDS:
    for task_name, task_dict in test_tasks.items():
        d, _, _ = adapt_and_evaluate(
            meta_model, task_dict, N_SHOT_STABILITY,
            adapt_steps=20, seed=seed
        )
        stability_scores[task_name].append(d)

print(f'\n5-shot stability across {len(STABILITY_SEEDS)} random seeds\n')
print(f'{"Task":<12}  {"Mean Dice":>10}  {"Std":>8}  {"Min":>8}  {"Max":>8}')
for task, scores in stability_scores.items():
    arr = np.array(scores)
    print(f'{task:<12}  {arr.mean():>10.4f}  '
          f'{arr.std():>8.4f}  {arr.min():>8.4f}  {arr.max():>8.4f}')

# Box plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([stability_scores[t] for t in test_tasks],
           labels=list(test_tasks.keys()))
ax.set_ylabel('Dice score')
ax.set_title(f'Result stability: {N_SHOT_STABILITY}-shot Reptile '
             f'across {len(STABILITY_SEEDS)} seeds')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('stability_analysis.png', dpi=150)
plt.show()